In [1]:
import pandas as pd
import gc

# Preparing dataset consisting of bus and train station coordinates

In [2]:
bus_data = pd.read_csv('../../data/A0008D - v_q_bus_stop (full).csv')
bus_location_data = pd.read_csv('../../data/correct_bus_location.csv')
bus_additional_data = pd.read_csv('../../data/A0008D - v_q_vls_marker (full).csv')

In [3]:
bus_consolidated = bus_data.merge(
   bus_location_data,
   how='outer',
   left_on='BUS_STOP_CD',
   right_on='BusStopCode'
)


bus_needed = bus_consolidated[[
   "BUS_STOP_CD",
   "BUS_STOP_NAM",
   "MRK_ID_NUM",
   "BusStopCode",
   "Description",
   "Latitude",
   "Longitude"
]].copy()


bus_needed = bus_needed.rename(columns={
   "BUS_STOP_NAM": "STATION_NAME",
   "Latitude": "LATITUDE",
   "Longitude": "LONGITUDE"
})


bus_needed = bus_needed.rename(columns={
   "BUS_STOP_NAM": "STATION_NAME",
   "Description": "LTA_DESCRIPTION",
   "Latitude": "LATITUDE",
   "Longitude": "LONGITUDE"
})
bus_needed["FINAL_STOP_CODE"] = bus_needed["BUS_STOP_CD"].fillna(bus_needed["BusStopCode"])
bus_needed["STATION_NAME"] = bus_needed["STATION_NAME"].fillna(bus_needed["LTA_DESCRIPTION"])
bus_needed['Travel_Type'] = 'BUS'


bus_needed_final = bus_needed[[
   "FINAL_STOP_CODE",
   "STATION_NAME",
   "MRK_ID_NUM",
   "LATITUDE",
   "LONGITUDE",
   "Travel_Type"
]].copy()


bus_additional_data["LATITUDE"] = bus_additional_data["LATITUDE_VAL"] / 3600000
bus_additional_data["LONGITUDE"] = bus_additional_data["LONGITUDE_VAL"] / 3600000


bus_additional_data_clean = bus_additional_data[[
   "MRK_ID_NUM",
   "LATITUDE",
   "LONGITUDE"
]].copy()


bus_needed_final = bus_needed_final.merge(
   bus_additional_data_clean,
   on="MRK_ID_NUM",
   how="left",
   suffixes=("", "_additional")
)


bus_needed_final["LATITUDE"] = bus_needed_final["LATITUDE"].fillna(
   bus_needed_final["LATITUDE_additional"]
)


bus_needed_final["LONGITUDE"] = bus_needed_final["LONGITUDE"].fillna(
   bus_needed_final["LONGITUDE_additional"]
)


bus_needed_final = bus_needed_final.drop(columns=[
   "LATITUDE_additional",
   "LONGITUDE_additional"
])


print("Remaining missing coords:",
     bus_needed_final["LATITUDE"].isna().sum())

Remaining missing coords: 0


In [4]:
train_data = pd.read_csv('../../data/mrt_station_coordinates.csv')
train_needed = train_data[['NODE_NAME', 'NODE_MRK_ID', 'LATITUDE', 'LONGITUDE']]
train_needed = train_needed.rename(columns={
   'NODE_NAME': 'STATION_NAME',
   'NODE_MRK_ID': 'MRK_ID_NUM'
})
train_needed['Travel_Type'] = 'TRAIN'
#train_needed.head(5)

In [5]:
bus_needed_final = bus_needed_final[[
   "STATION_NAME",
   "MRK_ID_NUM",
   "LATITUDE",
   "LONGITUDE",
   "Travel_Type"
]]


consolidated_stations = pd.concat(
   [bus_needed_final, train_needed],
   axis=0,
   ignore_index=True
)


consolidated_stations.tail(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
6271,Marine Parade,427.0,1.302865,103.905508,TRAIN
6272,Marine Terrace,428.0,1.306786,103.915316,TRAIN
6273,Siglap,429.0,1.309877,103.929879,TRAIN
6274,Bayshore,430.0,1.312841,103.941597,TRAIN
6275,Hume,336.0,1.354511,103.769104,TRAIN


In [6]:
# # memory checkpoint
%who
del bus_additional_data, bus_additional_data_clean,bus_consolidated, bus_data, bus_location_data, bus_needed, bus_needed_final, train_data, train_needed

bus_additional_data	 bus_additional_data_clean	 bus_consolidated	 bus_data	 bus_location_data	 bus_needed	 bus_needed_final	 consolidated_stations	 gc	 
pd	 train_data	 train_needed	 


# Data cleaning for pt_ride

In [7]:
df = pd.read_csv('../../data/A0008D - v_pt_ride_all (full).csv')

In [8]:
df['BIZ_DT'] = pd.to_datetime(df['BIZ_DT'])
df['ENTRY_DT'] = pd.to_datetime(df['ENTRY_DT'])
df['EXIT_DT'] = pd.to_datetime(df['EXIT_DT'])
df['ENTRY_TM'] = pd.to_datetime(
   df['ENTRY_DT'].astype(str) + ' ' + df['ENTRY_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
df['EXIT_TM'] = pd.to_datetime(
   df['EXIT_DT'].astype(str) + ' ' + df['EXIT_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)


### filtering to chosen day (12/2/2025)

In [9]:
df = df[df["ENTRY_DT"].dt.date == pd.Timestamp("2025-02-12").date()]

In [10]:
df2 = df[['BUS_SVC_NUM', 'CRD_NUM', 'DEST_LOC_ID_NUM', 'ENTRY_DT',
      'ENTRY_TM', 'EXIT_DT', 'EXIT_TM', 'JRNY_ID_NUM', 'ORIG_LOC_ID_NUM', 'RIDE_DISC_AMT', 'RIDE_DIST_KM_CNT',
      'RIDE_FARE_AMT', 'RIDE_ID_NUM', 'RIDE_MIN_CNT','PATRON_CATG_ID_NUM','TRNSPT_MODE_CD']]

In [11]:
df2 = df2.merge(consolidated_stations, how = 'left', left_on= 'DEST_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [12]:
df2.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [13]:
df2 = df2.merge(consolidated_stations, how = 'left', left_on= 'ORIG_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [14]:
df2.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)

In [15]:
# # # clear memory checkpoint
%who
del df
gc.collect()

consolidated_stations	 df	 df2	 gc	 pd	 


0

### Data cleaning for abt_ride

In [16]:
abt = pd.read_csv('../../data/A0008D - v_abt_pt_ride (full).csv')

C:\Users\shaog\AppData\Local\Temp\ipykernel_19544\2821325984.py:1: DtypeWarning: Columns (0: CRD_NUM) have mixed types. Specify dtype option on import or set low_memory=False.
  abt = pd.read_csv('../../data/A0008D - v_abt_pt_ride (full).csv')


In [17]:
df2['CRD_NUM'] = df2['CRD_NUM'].astype('object') # For consistency

In [18]:
abt['BIZ_DT'] = pd.to_datetime(abt['BIZ_DT'])
abt['ENTRY_DT'] = pd.to_datetime(abt['ENTRY_DT'])
abt['EXIT_DT'] = pd.to_datetime(abt['EXIT_DT'])
abt['ENTRY_TM'] = pd.to_datetime(
   abt['ENTRY_DT'].astype(str) + ' ' + abt['ENTRY_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
abt['EXIT_TM'] = pd.to_datetime(
   abt['EXIT_DT'].astype(str) + ' ' + abt['EXIT_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)

### filtering to chosen day (12/2/2025)

In [19]:
abt = abt[abt["ENTRY_DT"].dt.date == pd.Timestamp("2025-02-12").date()]

In [20]:
abt2 = abt[['BUS_SVC_NUM', 'CRD_NUM', 'DEST_LOC_ID_NUM', 'ENTRY_DT',
      'ENTRY_TM', 'EXIT_DT', 'EXIT_TM', 'JRNY_ID_NUM', 'ORIG_LOC_ID_NUM', 'RIDE_DISC_AMT', 'RIDE_DIST_KM_CNT',
      'RIDE_FARE_AMT', 'RIDE_ID_NUM', 'RIDE_MIN_CNT','PATRON_CATG_ID_NUM','TRNSPT_MODE_CD']]

In [21]:
# memory checkpoint
%who
del abt

abt	 abt2	 consolidated_stations	 df2	 gc	 pd	 


In [22]:
abt2 = abt2.merge(consolidated_stations, how = 'left', left_on= 'DEST_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [23]:
abt2.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [24]:
abt2 = abt2.merge(consolidated_stations, how = 'left', left_on= 'ORIG_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [25]:
abt2.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)
#abt2.head(5)

In [26]:
gc.collect()

0

### Combine pt_ride and abt_ride (stored as df2)

In [27]:
df2 = pd.concat(
   [df2, abt2],
   axis=0,
   ignore_index=True
)



In [28]:
df2.to_pickle('df2.pkl')
print('Saved df2.pkl')

Saved df2.pkl


In [29]:
# To load the dataframe in another file later
# df2 = pd.read_pickle('df2.pkl')

In [30]:
df3 = df2[df2["ENTRY_DT"].dt.date == pd.Timestamp("2025-02-12").date()] # Double check
df3 = df3.sort_values(["CRD_NUM", "ENTRY_DT", "ENTRY_TM"], ascending= [True, True, True]).reset_index(drop= True)

In [31]:
# clear memory checkpoint
%who
del df2, abt2
gc.collect()

abt2	 consolidated_stations	 df2	 df3	 gc	 pd	 


0

In [32]:
df3["next_orig_lat"] = df3.groupby("CRD_NUM")["ORIG_LATITUDE"].shift(-1)
df3["next_orig_lon"] = df3.groupby("CRD_NUM")["ORIG_LONGITUDE"].shift(-1)
df3["next_orig_station"] = df3.groupby("CRD_NUM")["ORIG_STATION_NAME"].shift(-1)

### Add walking distance 

In [33]:
walk_cache = pd.read_csv('../../data/walk_distance_cache.csv')

In [34]:
df3["pair_key"] = (
  df3["DEST_LATITUDE"].astype(str) + "_" +
  df3["DEST_LONGITUDE"].astype(str) + "_" +
  df3["next_orig_lat"].astype(str) + "_" +
  df3["next_orig_lon"].astype(str)
)
pairs = df3[
   [
       "pair_key",
       "DEST_STATION_NAME",
       "next_orig_station",
       "DEST_LATITUDE",
       "DEST_LONGITUDE",
       "next_orig_lat",
       "next_orig_lon"
   ]
].dropna(subset=[
   "DEST_LATITUDE",
   "DEST_LONGITUDE",
   "next_orig_lat",
   "next_orig_lon"
]).drop_duplicates("pair_key").copy()


In [35]:
df3 = (
   df3.drop(columns=["pair_key"], errors="ignore")
   .merge(
       walk_cache.drop(columns=["pair_key"], errors="ignore")[[
           "DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon", "walk_distance"
       ]].drop_duplicates(
           subset=["DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"]
       ),
       on=["DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"],
       how="left"
   )
)

In [36]:
gc.collect()

0

In [37]:
df3.to_pickle('df3.pkl')
print('Saved df3.pkl')

Saved df3.pkl


In [38]:
# memory checkpoint
%who
del df3, walk_cache, pairs
gc.collect()


consolidated_stations	 df3	 gc	 pairs	 pd	 walk_cache	 


0

### for normal laptops: 
restart kernel here and rerun top few cells to get consolidated_stations again

In [ ]:
# # memory checkpoint
%who
del bus_additional_data,bus_additional_data_clean,bus_consolidated,bus_data,bus_location_data,bus_needed,bus_needed_final,train_data,train_needed	 

# Data cleaning for pt_jrny (internal validity)

In [39]:
df4 = pd.read_csv('../../data/A0008D - v_pt_jrny_all (full).csv')

In [40]:
df4.dtypes

BIZ_DT                    str
CRD_NUM                 int64
JRNY_DEST_ID_NUM      float64
JRNY_DIST_KM_CNT      float64
JRNY_END_DT               str
JRNY_END_TM               str
JRNY_FARE_AMT         float64
JRNY_ID_NUM             int64
JRNY_ORIG_ID_NUM      float64
JRNY_ST_CD              int64
JRNY_START_DT             str
JRNY_START_TM             str
JRNY_TM_MIN_CNT       float64
PATRON_CATG_ID_NUM      int64
PAY_CD                  int64
TKT_TYP_CD              int64
TRNSPT_MODE_CD          int64
dtype: object

In [41]:
df4['BIZ_DT'] = pd.to_datetime(df4['BIZ_DT'])
df4['JRNY_START_DT'] = pd.to_datetime(df4['JRNY_START_DT'])
df4['JRNY_END_DT'] = pd.to_datetime(df4['JRNY_END_DT'])
df4['JRNY_START_TM'] = pd.to_datetime(
   df4['JRNY_START_DT'].astype(str) + ' ' + df4['JRNY_START_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
df4['JRNY_END_TM'] = pd.to_datetime(
   df4['JRNY_END_DT'].astype(str) + ' ' + df4['JRNY_END_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)

### filtering to chosen day (12/2/2025)

In [42]:
df4 = df4[df4["JRNY_START_DT"].dt.date == pd.Timestamp("2025-02-12").date()]

In [43]:
df5 = df4[['CRD_NUM', 'JRNY_DEST_ID_NUM', 'JRNY_START_DT',
      'JRNY_START_TM', 'JRNY_END_DT', 'JRNY_END_TM', 'JRNY_ORIG_ID_NUM', 'JRNY_DIST_KM_CNT',
      'JRNY_FARE_AMT', 'JRNY_ID_NUM', 'JRNY_TM_MIN_CNT','PATRON_CATG_ID_NUM','TRNSPT_MODE_CD']]

In [44]:
# # memory checkpoint
%who
del df4
gc.collect()

consolidated_stations	 df4	 df5	 gc	 pd	 


14

In [45]:
df5 = df5.merge(consolidated_stations, how = 'left', left_on= 'JRNY_DEST_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [46]:
df5.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [47]:
df5 = df5.merge(consolidated_stations, how = 'left', left_on= 'JRNY_ORIG_ID_NUM',
               right_on= 'MRK_ID_NUM')
df5.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)

In [48]:
gc.collect()

0

# Data cleaning for abt_jrny (internal validity)

In [49]:
abt3 = pd.read_csv('../../data/A0008D - v_abt_pt_jrny (full).csv')

C:\Users\shaog\AppData\Local\Temp\ipykernel_19544\3756125998.py:1: DtypeWarning: Columns (0: CRD_NUM) have mixed types. Specify dtype option on import or set low_memory=False.
  abt3 = pd.read_csv('../../data/A0008D - v_abt_pt_jrny (full).csv')


In [50]:
df5['CRD_NUM'] = df5['CRD_NUM'].astype('object') # For consistency

In [51]:
abt3['BIZ_DT'] = pd.to_datetime(abt3['BIZ_DT'])
abt3['JRNY_START_DT'] = pd.to_datetime(abt3['JRNY_START_DT'])
abt3['JRNY_END_DT'] = pd.to_datetime(abt3['JRNY_END_DT'])
abt3['JRNY_START_TM'] = pd.to_datetime(
   abt3['JRNY_START_DT'].astype(str) + ' ' + abt3['JRNY_START_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
abt3['JRNY_END_TM'] = pd.to_datetime(
   abt3['JRNY_END_DT'].astype(str) + ' ' + abt3['JRNY_END_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)

### Filtering to chosen day (12/2/2025)

In [52]:
abt3 = abt3[abt3["JRNY_START_DT"].dt.date == pd.Timestamp("2025-02-12").date()] 

In [53]:
abt4 = abt3[['CRD_NUM', 'JRNY_DEST_ID_NUM', 'JRNY_START_DT',
      'JRNY_START_TM', 'JRNY_END_DT', 'JRNY_END_TM', 'JRNY_ORIG_ID_NUM', 'JRNY_DIST_KM_CNT',
      'JRNY_FARE_AMT', 'JRNY_ID_NUM', 'JRNY_TM_MIN_CNT','PATRON_CATG_ID_NUM','TRNSPT_MODE_CD']]

In [54]:
# memory checkpoint
%who
del abt3

abt3	 abt4	 consolidated_stations	 df5	 gc	 pd	 


In [55]:
abt4 = abt4.merge(consolidated_stations, how = 'left', left_on= 'JRNY_DEST_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [56]:
abt4.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [57]:
abt4 = abt4.merge(consolidated_stations, how = 'left', left_on= 'JRNY_ORIG_ID_NUM',
               right_on= 'MRK_ID_NUM')
abt4.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)

In [58]:
gc.collect()

0

### combine pt_jrny and abt_jrny for internal validity

In [59]:
df5 = pd.concat(
   [df5, abt4],
   axis=0,
   ignore_index=True
)
df5 = df5.sort_values(["CRD_NUM", "JRNY_START_DT", "JRNY_START_TM"], ascending= [True, True, True]).reset_index(drop= True)

In [60]:
df5 = df5[df5["JRNY_START_DT"].dt.date == pd.Timestamp("2025-02-12").date()] # Double Check

In [61]:
df5.to_pickle('df5.pkl')
print('Saved df5.pkl')

Saved df5.pkl
